# Multivector Search Using the `turbo4` Datatype

As of [Qdrant 1.19](https://qdrant.tech/blog/qdrant-1.19.x/), Qdrant supports the `turbo4` datatype, which stores dense vectors on disk as 4-bit values per dimension instead of the 32 bits per dimension that `float32` uses, at some cost to recall.

This is possible thanks to [Google's TurboQuant](https://research.google/blog/turboquant-redefining-ai-efficiency-with-extreme-compression/) quantization technique: each vector is mathematically rotated so its information spreads evenly across all dimensions, which keeps the loss during compression low. Each rotated value is then stored as one of 16 levels, which fits in 4 bits and shrinks the vector to an eighth of its original size.

`turbo4` is inspired by this quantization method, but it is a **standalone data type** that you can quantize further. For example, you can combine `turbo4` with 1-bit TurboQuant quantization.

`turbo4` also works with multivector representations, and that is what this notebook demonstrates.

## Setup

We will use [Qdrant Cloud Inference](https://qdrant.tech/documentation/cloud/inference/) to generate embeddings and [Qdrant collections](https://qdrant.tech/documentation/concepts/collections/) to store them, so the `qdrant-client` package is the only Qdrant dependency we need.

We will also use `huggingface-hub` and `polars` to download and process the dataset.

In [1]:
! pip -q install qdrant-client huggingface-hub polars httpx

## Dataset

We will download the [`McAuley-Lab/Amazon-Reviews-2023`](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023) dataset, specifically its `Pet_Supplies` category, and load it with Polars.

In [2]:
from huggingface_hub import snapshot_download

path = snapshot_download(
    "McAuley-Lab/Amazon-Reviews-2023",
    repo_type="dataset",
    allow_patterns=["raw/meta_categories/meta_Pet_Supplies.jsonl"],
)

print(path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

/root/.cache/huggingface/hub/datasets--McAuley-Lab--Amazon-Reviews-2023/snapshots/2b6d039ed471f2ba5fd2acb718bf33b0a7e5598e


In [3]:
import polars as pl

df = pl.read_ndjson(path + "/raw/meta_categories/meta_Pet_Supplies.jsonl", ignore_errors=True, n_rows=200_000)

## Data Preparation

We will keep only a few columns from the dataframe:

- `title`: the product name, embedded with [BM25](https://qdrant.tech/documentation/inference/inference-bm25/) as a sparse vector.
- `description`: the product description, embedded with [ColBERT](https://qdrant.tech/articles/late-interaction-models/), a late-interaction text embedding model that produces a multivector per document.
- `images`: the product images, embedded with the `qdrant/clip-vit-b-32-vision` model.
- `details` and `price`, kept as payload metadata.

In [18]:
df = df.drop_nans()
df = df.drop_nulls()
df = df.select(["title", "description", "images", "price", "details"])
df = df.filter((pl.col("images").list.len() > 0) & (pl.col("description").list.len() > 0))
print(f"Dataset size: {df.height}")

Dataset size: 49310


# Create a Qdrant Collection

First, create a [Qdrant cluster](https://qdrant.tech/documentation/cloud/create-cluster/#standard-clusters), save its URL and API key, and use them to instantiate the Qdrant client.

In [31]:
from qdrant_client import AsyncQdrantClient
from getpass import getpass

client = AsyncQdrantClient(
    url=getpass("Qdrant URL: "),
    api_key=getpass("Qdrant API key: "),
    timeout=600,
    cloud_inference=True,
)

Qdrant URL: ··········
Qdrant API key: ··········


Now let's create a collection with the different vectors we will search over, using `turbo4` as the datatype for the dense vectors.

Note that the `description` vector sets `multivector_config` with `MAX_SIM` as the comparator. This tells Qdrant to score each document by its best-matching token pair, which is how ColBERT's late-interaction retrieval works.

In [ ]:
from qdrant_client import models

await client.create_collection(
    collection_name="pet_supplies",
    vectors_config={
        "description": models.VectorParams(
            size=96,
            distance=models.Distance.COSINE,
            multivector_config=models.MultiVectorConfig(
              comparator=models.MultiVectorComparator.MAX_SIM
            ),
            datatype=models.Datatype.TURBO4,
        ),
        "image": models.VectorParams(
            size=512,
            distance=models.Distance.COSINE,
            datatype=models.Datatype.TURBO4,
        ),
    },
    sparse_vectors_config={
        "title": models.SparseVectorParams(modifier=models.Modifier.IDF)
    }
)

## Upload Data

With the collection created, we can upload the data and let Cloud Inference embed it server-side, so we never have to load an embedding model locally.

In [ ]:
import uuid
from typing import Any


def get_image(img_dict: dict[str, Any]) -> str:
    try:
      return img_dict["large"]
    except KeyError:
      return img_dict[next(iter(img_dict))]

for batch in df.iter_slices(1):
  points = [
      models.PointStruct(
        id=str(uuid.uuid4()),
        vector={
            "description": models.Document(
                text="\n".join(row["description"]),
                model="answerdotai/answerai-colbert-small-v1",
            ),
            "image": models.Image(
                image=get_image(row["images"][0]),
                model="qdrant/clip-vit-b-32-vision"
            ),
            "title": models.Document(
                text=row["title"],
                model="qdrant/bm25"
            )
        },
        payload={
            "price": row["price"],
            "details": row["details"],
            "title": row["title"],
            "image": get_image(row["images"][0]),
            "description": "\n".join(row["description"]),
        },
      )
      for row in batch.iter_rows(named=True)
  ]
  await client.upsert(collection_name="pet_supplies", points=points)

## Querying

Now let's query the data. We will [prefetch](https://qdrant.tech/documentation/concepts/hybrid-queries/#query-api) candidates using either the image or the title vector, then re-score them with the ColBERT `description` vector through late interaction.

In [47]:
query = "Orijen dry cat food"
image_query = models.Document(
    text=query,
    model="qdrant/clip-vit-b-32-text"
)
title_query = models.Document(
    text=query,
    model="qdrant/bm25",
)
colbert_query = models.Document(
    text=query,
    model="answerdotai/answerai-colbert-small-v1"
)

In [51]:
response = await client.query_points(
    collection_name="pet_supplies",
    prefetch=models.Prefetch(
        query=image_query,
        using="image"
    ),
    query=colbert_query,
    limit=1,
    with_payload=True,
    using="description"
)

In [52]:
result = response.points[0]

print(result.payload["title"])

ORIJEN® Dry Adult Cat Food, Grain Free, Premium, High Protein, Fresh & Raw Animal Ingredients, Guardian 8, 10lb


In [ ]:
import httpx

from io import BytesIO
from PIL import Image

image_url = result.payload["image"]


async with httpx.AsyncClient() as http_client:
  img_resp = await http_client.get(image_url)
  img_resp.raise_for_status()
  content = img_resp.content

Image.open(BytesIO(content))

In [55]:
response_bm25 = await client.query_points(
    collection_name="pet_supplies",
    prefetch=models.Prefetch(
        query=title_query,
        using="title"
    ),
    query=colbert_query,
    limit=10,
    with_payload=True,
    using="description"
)

In [ ]:
from textwrap import wrap

result_bm25 = response_bm25.points[0]

print(result_bm25.payload["title"])
print()
print("\n".join(wrap(result_bm25.payload["description"])))